In [ ]:
! wget https://github.com/marcin119a/data/raw/refs/heads/main/data_gsn.zip
! unzip data_gsn.zip &> /dev/null
! rm data_gsn.zip

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.io as io
from torch.utils.data import Dataset, DataLoader
from copy import deepcopy


import pandas as pd
import numpy as np

from plot import *

In [ ]:
FIGSIZE = (12, 10)
PX_SIZE = (1280, 720)

N_PAIR_CLASSES = 15
N_CLS_CLASSES = 135
N_CNT_CLASSES = 6
TRAIN_SIZE = 9000
DATASET_SIZE = 10000

N_EPOCHS = 100
TRAIN_BATCH_SIZE = 64
VAL_BATCH_SIZE = 1000

torch.manual_seed(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32

### Map mask to class

I have to be able to map the original amounts (6 classes, exactly two values are greater than 0 and sum to 10) into 135 classes.

This is a pretty simple method:
- first I encode the binary classes in "lexicographical" order (for example (0 0 4 0 6 0) is encoded as (0 0 1 0 1 0) and is before (0 0 0 1 1 0) but after (0 0 1 1 0 0)). I can do it with formula $f(i_0, i_2) = i_0 \cdot (9 - i_0) / 2 + i_1 - 1$, where $i_0$ is the index of the first non zero element and $i_1$ is the index of the second non zero element.
- then I multiply the first non zero element by 9 and add the value at the second non zero element minus 1 (then it's in range 0-8).

This way i can also easily tell which pair it was - by deviding by 9 with integer division.

I also implemented some tests to be sure I don't hava an off by one.

In [ ]:
from model import (
    amounts_to_class, 
    class_to_pair_encoding, 
    test_class_translation
)
test_class_translation()

### Model definition

In [ ]:
from model import Model

model = Model()

### Data augmentation

I have implemented 3 data augmentations:
- horizontal flip with probability 0.5
- vertical flip with probability 0.5
- rotation by 90 degrees with probability 2/3

Rotation has probability 2/3 because it can be done in 2 ways (clockwise and counterclockwise) and I decide which way with equal probability. So all possibilities (no rotation, clockwise, counterclockwise) have equal probability of 1/3.

In [ ]:

from dataloading import (
    Augmentor,
    HorizontalFlipAugmentor,
    VerticalFlipAugmentor,
    Rotation90Augmentor
)

### Dataset

Because the dataset is quite small and the pictures have a low resolution, I decided to load the entire dataset into gpu vram at once. It easily fits and is much faster than loading on the fly. Not too professional, but works well for a small project like this.

Inside the Dataset class, I include the augmentations. I switch them off for validation.

In [ ]:
from dataloading import ImageDataset

train_data = ImageDataset(
    range(TRAIN_SIZE),
    augment=True,
    device=device,
    dtype=dtype
)
val_data = ImageDataset(
    range(TRAIN_SIZE, DATASET_SIZE),
    augment=False,
    device=device,
    dtype=dtype
)

### Check augmentation correctness

It's good to visually check if the augmentations are correct.

In [ ]:
horizontal = HorizontalFlipAugmentor(1)
vertical = VerticalFlipAugmentor(1)
rotation90 = Rotation90Augmentor(1)

image, _, amounts = val_data[2]

image = image.to('cpu')
amounts = amounts.to('cpu')

h_img, h_amounts = horizontal(image, amounts)
v_img, v_amounts = vertical(image, amounts)
r1_img, r1_amounts = rotation90(image, amounts)
r2_img, r2_amounts = rotation90(image, amounts)
while torch.equal(r2_img, r1_img):
    r2_img, r2_amounts = rotation90(image, amounts)

show_image(image, "Original image", (4, 4))
print("Original label:", amounts)
show_image(h_img, "Horizontal flip", (4, 4))
print("Horizontal flip label:", h_amounts)
show_image(v_img, "Vertical flip", (4, 4))
print("Vertical flip label:", v_amounts)
show_image(r1_img, "Rotation 1", (4, 4))
print("Rotation 1 flip label:", r1_amounts)
show_image(r2_img, "Rotation 2", (4, 4))
print("Rotation 2 flip label:", r2_amounts)

### Exploratory Data Analysis

first I we should check the distribution of classes in the dataset.

In [ ]:
how_many = torch.zeros(N_CLS_CLASSES, dtype=torch.int32)

train_data.augment = False

for _, cls, _ in train_data:
    how_many[cls.to("cpu")] += 1

train_data.augment = True


plot_bar(
    how_many.numpy(),
    title="Number of samples per class in training set",
    xlabel="Class",
    ylabel="Number of samples"
)

We can see that the distribution is pretty balanced. However, some pairs do not occur at all. Augmentations I apply will make the dataset even more balanced as every triangle direction can be transformed into any other.

We should also check if the distribution in the regression task is balanced. To that I will count how many times each number from 1 to 9 is present in the dataset.

In [ ]:
occurences = torch.zeros(10, dtype=torch.float32)

train_data.augment = False

for _, cls, amounts in train_data:
    amounts = amounts.to('cpu')
    i0, i1 = torch.nonzero(amounts)
    occurences[int(amounts[i0])] += 1
    occurences[int(amounts[i1])] += 1

train_data.augment = True

plot_bar(
    occurences.numpy(),
    title=f"Distribution of counts in training set",
    xlabel="Count",
    ylabel="Number of occurences"
)

We can see that the number of occurences form a uniform distribution, which is good. This also explains the "gaps" in the classification task - in the datase there is never a split (1, 9) of the 10 geometric shapes, so classes corresponding to that split do not occur at all.

I've also previously checked that this is true for all 6 classes. But I don't want this notebook to be too long.

### Metrics

I implemented all metrics that were specified in the task description. I decided to implement them as classes which keep track of the values. Each metric class also has a method to plot it.

In [ ]:
from metrics import (
    Top1Accuracy,
    PerPairAccuracy,
    MacroF1Score,
    RMSEPerClass,
    RMSE,
    MAEPerClass,
    MAE
)

### Trainer

I implemented a trainer class instead of just a function or loop. This way I can easily launch multiple trainings with different parameters to compare them. It's also just a habit.

I aslo implemented loss functions. The loss for multitask learning can be easily used to train classification only model by setting lambda to 0. I had to implement a separate class for regression loss. It looks a bit weird but it enables me to use the same trainer.

In [ ]:
from trainer import Trainer, MultitaskLoss, RegressionLoss

### Training

In [ ]:
multitask_loss = MultitaskLoss(1.0)
classification_loss = MultitaskLoss(0.0)
regression_loss = RegressionLoss(1.0)

train_dataloader = DataLoader(
    train_data,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=ImageDataset.collate_fn
)
val_dataloader = DataLoader(
    val_data,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=ImageDataset.collate_fn
)

#### Multitask training

In [ ]:
model_multitask = torch.compile(deepcopy(model).to(device, dtype))
optimizer = optim.Adam(
    model_multitask.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)

multitask_eval_metrics = [
    Top1Accuracy(),
    MacroF1Score(),
    PerPairAccuracy(),
    RMSEPerClass(),
    MAEPerClass(),
    RMSE(),
    MAE(),
]

multitask_trainer = Trainer(
    model=model_multitask,
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    loss_fn=multitask_loss,
    eval_metrics=multitask_eval_metrics,
    optimizer=optimizer,
    n_epochs=N_EPOCHS,
    patience=10
)

multitask_trainer.train()

In [ ]:
make_metric_plots(
    multitask_trainer,
    multitask_eval_metrics,
    title="Multitask training",
    use_cls_metrics=True,
    use_cnt_metrics=True,
    grid_shape=(3, 3)
)

#### Classification only training

In [ ]:
model_cls = torch.compile(deepcopy(model).to(device, dtype))
model_cls.forward = model_cls.forward_cls
optimizer = optim.Adam(
    model_cls.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)

cls_eval_metrics = [
    Top1Accuracy(),
    MacroF1Score(),
    PerPairAccuracy(),
]

cls_trainer = Trainer(
    model=model_cls,
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    loss_fn=classification_loss,
    eval_metrics=cls_eval_metrics,
    optimizer=optimizer,
    n_epochs=N_EPOCHS,
    patience=10
)

cls_trainer.train()

In [ ]:
make_metric_plots(
    cls_trainer,
    cls_eval_metrics,
    title="Classification only training",
    use_cls_metrics=True,
    use_cnt_metrics=False,
    grid_shape=(2, 2)
)

#### Regression only training

In [ ]:
model_cnt = torch.compile(deepcopy(model).to(device, dtype))
model_cnt.forward = model_cnt.forward_cnt
optimizer = optim.Adam(
    model_cnt.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)

cnt_eval_metrics = [
    RMSE(),
    MAE(),
    RMSEPerClass(),
    MAEPerClass(),
]

cnt_trainer = Trainer(
    model=model_cnt,
    train_dataloader=train_dataloader,
    eval_dataloader=val_dataloader,
    loss_fn=regression_loss,
    eval_metrics=cnt_eval_metrics,
    optimizer=optimizer,
    n_epochs=N_EPOCHS,
    patience=10
)

cnt_trainer.train()

In [ ]:
make_metric_plots(
    cnt_trainer,
    cnt_eval_metrics,
    title="Regression only training",
    use_cls_metrics=False,
    use_cnt_metrics=True,
    grid_shape=(3, 2)
)

### Result analysis

Let's check the confusion matrix for the classification task. I will use the model trained for multitask learning as it had the best accuracy.

In [ ]:
from sklearn.metrics import confusion_matrix

model_multitask.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in val_dataloader:
        x, y_cls, _ = batch
        pred_cls, _ = model_multitask(x)

        all_preds.append(pred_cls)
        all_targets.append(y_cls)

all_preds = torch.argmax(torch.cat(all_preds), dim=-1).cpu().numpy()
all_targets = torch.cat(all_targets).cpu().numpy()

cm = confusion_matrix(all_targets, all_preds, labels=np.arange(N_CLS_CLASSES))

fig = confusion_matrix_fig(
    cm, N_CLS_CLASSES, title="Confusion Matrix - Multitask Model"
)

fig.update_layout(width=PX_SIZE[0], height=PX_SIZE[1])
fig.show()

I will check some missclassified examples. I also want to check how many errors were caused by predicting a good pair but with wrong number of first shape occurences.

In [ ]:
class_to_shapes = {}

for i in range(5):
    for j in range(i + 1, 6):
        for k in range(1, 10):
            # k times elemnt i and 10 - k times element j
            t = torch.zeros(6)
            t[i] = k
            t[j] = 10 - k
            cls = amounts_to_class(t)
            class_to_shapes[cls.item()] = (i, j)


errors = all_preds != all_targets


indexes = np.arange(len(all_targets))
error_indexes = indexes[errors]

n_correct_pairs = 0

for i in error_indexes:
    img, cls, amounts = val_data[i]
    pred_cls = all_preds[i]
    true_pair = class_to_shapes[cls.item()]
    pred_pair = class_to_shapes[pred_cls]
    if true_pair == pred_pair:
        n_correct_pairs += 1

print(f"Number of misclassified samples: {len(error_indexes)}")
print(f"Number of misclassified samples with correct shape pairs: {n_correct_pairs}")


for i in range(3):
    idx = error_indexes[i]
    img, cls, amounts = val_data[idx]
    pred_cls = all_preds[idx]
    true_pair = class_to_shapes[cls.item()]
    pred_pair = class_to_shapes[pred_cls]
    show_image(
        img.to('cpu'),
        title=f"True shape {true_pair}, "
        f"Predicted shapes {pred_pair}",
        figsize=(4, 4)
    )

We can see that for most of the misclassified samples, my model actually predicted a correct pair, but with an incorrect split of elements.